In [ ]:
# ==========================================
# 1. 安裝必要的套件 (每次重開 Colab 都要跑)
# ==========================================
!pip install -q sentence-transformers beir


In [ ]:
# ==========================================
# 2. 載入資料與設定
# ==========================================
import logging
from beir import util, LoggingHandler
from beir.datasets.data_loader import GenericDataLoader
from sentence_transformers import SentenceTransformer, losses, models, InputExample
from sentence_transformers import evaluation
from torch.utils.data import DataLoader
import pathlib, os
import random

# 設定 Log
logging.basicConfig(format='%(asctime)s - %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S',
                    level=logging.INFO,
                    handlers=[LoggingHandler()])

# 下載並載入 NFCorpus 資料集
dataset = "nfcorpus"
url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip".format(dataset)
out_dir = os.path.join(pathlib.Path("").parent.absolute(), "datasets")
data_path = util.download_and_unzip(url, out_dir)

corpus, queries, qrels = GenericDataLoader(data_path).load(split="train")
# 載入開發集 (用來觀察訓練成效)
dev_corpus, dev_queries, dev_qrels = GenericDataLoader(data_path).load(split="dev")

print(f"Loaded {len(corpus)} documents and {len(queries)} training queries.")

In [ ]:
# ==========================================
# 3. 準備訓練資料 (關鍵優化點)
# ==========================================
train_examples = []

# 我們使用 [Query, Positive Document] 的配對
# 配合 MultipleNegativesRankingLoss，Batch 裡的其他文章會自動變成負樣本
# 這就是為什麼 Batch Size 要大的原因！
for qid in qrels:
    if qid not in queries: continue
    query_text = queries[qid]
    
    for doc_id, rel in qrels[qid].items():
        if rel > 0: # 只取相關的 (Positive)
            if doc_id not in corpus: continue
            doc_text = corpus[doc_id].get("title", "") + " " + corpus[doc_id].get("text", "")
            train_examples.append(InputExample(texts=[query_text, doc_text]))

print(f"Prepared {len(train_examples)} training pairs.")

In [ ]:
# ==========================================
# 4. 初始化模型 (不凍結任何層！)
# ==========================================
model_name = "sentence-transformers/all-MiniLM-L6-v2"
word_embedding_model = models.Transformer(model_name, max_seq_length=256) # 增加長度可能會有幫助
pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension())
model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

print("Model loaded. Full fine-tuning enabled (No layers frozen).")

# ==========================================
# 5. 設定訓練參數 (奪冠關鍵！)
# ==========================================

# ★ 關鍵 1: Batch Size 加大
# T4 GPU 應該可以跑 32，如果跳出 OOM (Out of Memory) 錯誤，請改回 16
train_batch_size = 32 

# ★ 關鍵 2: DataLoader
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=train_batch_size)

# ★ 關鍵 3: Loss Function
# 使用 MultipleNegativesRankingLoss，這是訓練檢索模型最強的 Loss 之一
train_loss = losses.MultipleNegativesRankingLoss(model=model)

# 準備 Evaluator (讓我們可以看到訓練過程中的進步)
# 這裡我們用 dev set 來評估，避免過度擬合
evaluator = evaluation.InformationRetrievalEvaluator(dev_queries, dev_corpus, dev_qrels, name='nfcorpus-dev')

In [ ]:
# ==========================================
# 6. 開始訓練
# ==========================================
num_epochs = 4 # ★ 關鍵 4: 增加訓練輪數
warmup_steps = int(len(train_dataloader) * num_epochs * 0.1) # 10% Warmup

print(f"Starting training for {num_epochs} epochs with batch size {train_batch_size}...")

# 這裡移除了凍結層的程式碼，直接讓它全速運轉
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    epochs=num_epochs,
    evaluation_steps=1000, # 每 1000 步評估一次
    warmup_steps=warmup_steps,
    output_path='finetuned_senBERT_v3_kaggle', # 存到新的資料夾
    save_best_model=True, # 自動儲存分數最好的那一版
    show_progress_bar=True
)

print("Training complete! Model saved to 'finetuned_senBERT_v3_kaggle'")

In [ ]:
# ==========================================
# 7. 打包下載 (防斷線)
# ==========================================
!zip -r finetuned_kaggle_model.zip finetuned_senBERT_v3_kaggle
from google.colab import files
files.download('finetuned_kaggle_model.zip')